# 02 - Pathway Score Computation

This notebook computes pathway activity scores from gene expression data.

**Method**: For each of 7 biological pathways, we:
1. Z-score normalize expression values per gene across all samples within the cohort
2. Compute the pathway score as the mean of normalized constituent gene expression
3. Add the engineered ratio feature: `Ratio_Prolif_Apop = Proliferation - Apoptosis`

**Pathways** (45 genes total):
- Proliferation (10 genes): MKI67, AURKA, BIRC5, CCNB1, MYBL2, PLK1, BUB1, CDC20, MCM2, PCNA
- Estrogen Response (6): ESR1, PGR, FOXA1, GATA3, XBP1, CA12
- Immune Response (7): CD8A, CD4, PDCD1, CD274, CTLA4, FOXP3, IL2RA
- Invasion/EMT (7): VIM, CDH2, FN1, MMP9, TWIST1, SNAI1, ZEB1
- Apoptosis (6): CASP3, CASP8, BAX, BAK1, BAD, TP53
- HER2 Signaling (5): ERBB2, GRB7, PGAP3, MIEN1, STARD3
- Angiogenesis (4): VEGFA, KDR, FLT1, FGF2

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_tcga_feature_matrix, load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, filter_outcome
from src.features import PATHWAY_DEFINITIONS, compute_pathway_scores, add_ratio_features

## 1. Pathway Definitions

In [ ]:
print("Biological Pathways:")
print("=" * 50)
total_genes = 0
for name, genes in PATHWAY_DEFINITIONS.items():
    print(f"  {name} ({len(genes)} genes): {', '.join(genes)}")
    total_genes += len(genes)
print(f"\nTotal unique genes: {total_genes}")

## 2. Verify TCGA Feature Matrix

The TCGA feature matrix was previously computed. We verify its structure.

In [ ]:
tcga_fm = load_tcga_feature_matrix('../data/processed/02_tcga_feature_matrix.csv')

pathway_cols = [c for c in tcga_fm.columns if c.startswith('Pathway_') or c.startswith('Ratio_')]
print(f"\nPathway feature columns ({len(pathway_cols)}): {pathway_cols}")
print(f"\nPathway score statistics:")
tcga_fm[pathway_cols].describe().round(3)

In [ ]:
# Distribution of pathway scores
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(pathway_cols):
    ax = axes[i // 4, i % 4]
    tcga_fm[col].hist(bins=30, ax=ax, alpha=0.7)
    ax.set_title(col.replace('Pathway_', '').replace('Ratio_', 'Ratio: '), fontsize=10)
    ax.set_xlabel('Score')
plt.suptitle('TCGA Pathway Score Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Compute GSE96058 Pathway Scores

**Requires**: Downloaded expression data (`data/raw/GSE96058_gene_expression.csv`).  
Run `cd data/raw && bash download_data.sh` if not yet downloaded.

In [ ]:
import os

gse_exp_path = '../data/raw/GSE96058_gene_expression.csv'

if os.path.exists(gse_exp_path):
    # Load expression data (genes as rows, samples as columns -> transposed)
    gse_exp = load_gse96058_expression(gse_exp_path)
    
    # Z-score normalize per gene across samples
    print("\nZ-score normalizing expression data...")
    gse_exp_norm = zscore_normalize(gse_exp)
    
    # Compute pathway scores
    print("\nComputing pathway scores...")
    gse_pathway = compute_pathway_scores(gse_exp_norm)
    gse_pathway = add_ratio_features(gse_pathway)
    gse_pathway.insert(0, 'sample_id', gse_exp['sample_id'].values)
    
    print(f"\nGSE96058 pathway scores shape: {gse_pathway.shape}")
    print(gse_pathway.describe().round(3))
else:
    print(f"Expression data not found at {gse_exp_path}")
    print("Run: cd ../data/raw && bash download_data.sh")
    print("\nSkipping GSE96058 pathway computation.")

## 4. Summary

The 8 pathway-level features (7 pathway scores + 1 ratio) capture biologically meaningful axes of variation in breast cancer gene expression.

In [ ]:
print("=" * 50)
print("PATHWAY FEATURE SUMMARY")
print("=" * 50)
print(f"Total pathways: 7")
print(f"Total genes across pathways: 45")
print(f"Engineered features: 1 (Ratio_Prolif_Apop)")
print(f"Total pathway features: 8")
print(f"\nTCGA samples with pathway scores: {len(tcga_fm)}")